# Lab 2 — Visualizing Transformer Dataflow

**Course**: *Dataflow Architectures for AI: From Principles to Practice*
**Week**: 2 — Dataflow Execution Models
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research / Columbia Engineering)
**Runtime**: Google Colab, GPU or CPU runtime — this lab is mostly static analysis
**Expected time**: 90–120 minutes

---

## Learning objectives

By the end of this lab you will be able to:
1. Trace a transformer block into an **FX graph** using `torch.fx.symbolic_trace`.
2. Visualize the resulting computational graph as nodes and edges.
3. Annotate each node with **compute cost (FLOPs)** and **memory footprint (bytes)**.
4. Classify nodes as **compute-bound**, **memory-bound**, or **cheap** using arithmetic intensity.
5. Predict which nodes would become scheduling bottlenecks on a generic dataflow accelerator.

## Why this matters

A dataflow accelerator's compiler does not see your PyTorch code. It sees a graph — nodes (operators), edges (tensors), dependencies. **This lab is the first time you will look at your model the way the hardware does.**

On Spyre (and TPU and Groq), the compiler decides fusion boundaries, tiling, scheduling, and memory placement all from this graph. If you can't read the graph, you can't reason about the hardware.

## Prerequisites

- Lab 1 complete.
- You've read/watched the Week 2 lecture material on dataflow execution principles and FX graphs.

## Grading

Same hybrid model as Lab 1:
- **Auto-graded checkpoints (60%)**: assertion cells verify graph structure, FLOP and byte counts, classification.
- **Peer-reviewed analysis (40%)**: the final section asks you to predict scheduling bottlenecks on a dataflow target.


## Part 0 — Setup

`torch.fx` is PyTorch's mechanism for capturing models as graphs. It works on *symbolic-traceable* modules — roughly, modules whose `forward` doesn't contain data-dependent Python control flow.

Modern LLMs like the full Qwen 2.5 model *are not* cleanly fx-traceable because of KV cache handling, position embedding math, and generation loops. For this lab we use a **simplified but faithful transformer block** that mirrors Qwen's structure (pre-norm, Q/K/V projections, scaled-dot-product attention, SwiGLU-style FFN) while remaining fully traceable.

> In Week 3 you'll see `torch.export` and `torch.compile`'s Dynamo, which *do* handle the full model. FX is the pedagogically cleaner starting point.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.fx
from collections import OrderedDict

torch.manual_seed(0)

print(f"torch = {torch.__version__}")
print(f"device = {'cuda' if torch.cuda.is_available() else 'cpu'}")


In [ ]:
class TinyTransformerBlock(nn.Module):
    """
    A traceable transformer block. Dimensions roughly match Qwen 2.5 0.5B:
        hidden_dim = 896, num_heads = 14, head_dim = 64, ffn_dim = 4864.

    For lab tractability we use smaller dims by default. The ratios are preserved
    so the FLOP / memory analysis transfers directly to the real model.

    Architecture: pre-norm, scaled-dot-product attention (no GQA for simplicity),
    SwiGLU-adjacent FFN (gated), residual adds.
    """
    def __init__(self, hidden_dim=512, num_heads=8, ffn_dim=2048):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.ffn_dim    = ffn_dim

        # Attention projections
        self.q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.o_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)

        # Norms
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)

        # FFN: gated ("SwiGLU-ish" — two up-projections, one down)
        self.ffn_gate = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.ffn_up   = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.ffn_down = nn.Linear(ffn_dim, hidden_dim, bias=False)

    def forward(self, x):
        # x: [batch, seq, hidden]
        B, S, H = x.shape[0], x.shape[1], x.shape[2]
        D = self.head_dim
        N = self.num_heads

        # --- Attention block (pre-norm) ---
        h = self.ln1(x)
        q = self.q_proj(h).reshape(B, S, N, D).transpose(1, 2)   # [B, N, S, D]
        k = self.k_proj(h).reshape(B, S, N, D).transpose(1, 2)
        v = self.v_proj(h).reshape(B, S, N, D).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(D)  # [B, N, S, S]
        attn   = torch.softmax(scores, dim=-1)
        out    = torch.matmul(attn, v)                               # [B, N, S, D]
        out    = out.transpose(1, 2).reshape(B, S, H)
        out    = self.o_proj(out)
        x      = x + out

        # --- FFN block (pre-norm, gated) ---
        h = self.ln2(x)
        gate = torch.nn.functional.silu(self.ffn_gate(h))
        up   = self.ffn_up(h)
        h    = gate * up                                              # gated activation
        h    = self.ffn_down(h)
        x    = x + h
        return x


# Instantiate and smoke-test
HIDDEN_DIM = 512
NUM_HEADS  = 8
FFN_DIM    = 2048
BATCH      = 2
SEQ        = 128

block = TinyTransformerBlock(HIDDEN_DIM, NUM_HEADS, FFN_DIM).eval()
dummy = torch.randn(BATCH, SEQ, HIDDEN_DIM)
out   = block(dummy)
assert out.shape == dummy.shape
print(f"Block output shape: {out.shape}")
print(f"Parameters: {sum(p.numel() for p in block.parameters()):,}")


## Part 1 — Trace the block with `torch.fx`

`torch.fx.symbolic_trace` walks through `forward()` with *proxy* tensors, recording every operation that touches a proxy. The output is a `GraphModule` containing a `Graph` of `Node` objects.

Each node has:
- `node.op` — one of `placeholder`, `call_function`, `call_method`, `call_module`, `output`.
- `node.target` — the function/method/module being called.
- `node.args`, `node.kwargs` — its inputs.
- `node.meta` — a dict where you can attach your own metadata (we'll use this in Part 3).

**Your task**:
1. Trace `block` into a `GraphModule`.
2. Print the tabular representation of the graph.
3. Count how many nodes fall into each `op` category.


In [ ]:
# TODO 1.1 — trace the block with torch.fx.symbolic_trace into `gm`
gm = None  # REPLACE

# TODO 1.2 — print the tabular view so you can see the graph structure
# Hint: gm.graph.print_tabular()

# TODO 1.3 — count nodes per op type, store in `op_counts` (dict[str, int])
op_counts = {}  # REPLACE

print("\nNode counts by op type:")
for k, v in op_counts.items():
    print(f"  {k:20s}: {v}")
print(f"  total: {sum(op_counts.values())}")


In [ ]:
# --- Checkpoint 1 -------------------------------------------------------
assert isinstance(gm, torch.fx.GraphModule), "gm must be a GraphModule from torch.fx.symbolic_trace"
assert "placeholder" in op_counts and op_counts["placeholder"] >= 1
assert "output" in op_counts and op_counts["output"] == 1
assert "call_module" in op_counts, "Should have call_module nodes (Linears, LayerNorms)"
assert op_counts["call_module"] >= 8, "Expected >= 8 submodule calls (q/k/v/o/ln1/ln2/ffn_gate/ffn_up/ffn_down)"
total = sum(op_counts.values())
assert 20 <= total <= 60, f"Graph size looks off: {total} nodes"
print(f"[OK] Traced block into a {total}-node graph.")


## Part 2 — Visualize the graph

Reading the tabular print is fine for debugging but doesn't give you a feel for the *dataflow shape*. Let's draw it.

We'll use `networkx` (ships with Colab) to lay out the graph and `matplotlib` to render it. We color nodes by `op` type so `call_module` (parameterized ops like Linear) pop visually — these are the nodes the dataflow compiler will spend the most effort on.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# TODO 2.1 — build a networkx DiGraph from the FX graph.
#             Each FX node becomes a graph node (keyed by node.name).
#             For each FX node, add edges FROM its input nodes (node.all_input_nodes).
G = nx.DiGraph()
# YOUR CODE HERE

# Visualization (provided — you don't need to modify below this line)
TERRACOTTA = "#B85C2C"
CREAM      = "#FAF6F2"
SECONDARY  = "#5C5C5C"

color_for_op = {
    "placeholder":    "#A8C5A8",   # soft green
    "call_module":    TERRACOTTA,  # course accent
    "call_function":  "#5C5C5C",
    "call_method":    "#BFBFBF",
    "output":         "#2F2F2F",
    "get_attr":       "#CDCDCD",
}
op_of  = {n.name: n.op for n in gm.graph.nodes}
labels = {n.name: n.name for n in gm.graph.nodes}
colors = [color_for_op.get(op_of[name], "#999") for name in G.nodes]

try:
    pos = nx.nx_agraph.graphviz_layout(G, prog="dot")
except Exception:
    # Fallback if graphviz isn't available: use a spring layout
    pos = nx.spring_layout(G, seed=0, k=0.9, iterations=80)

fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx_edges(G, pos, ax=ax, edge_color=SECONDARY, arrows=True, arrowsize=10, alpha=0.6)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors, node_size=900, alpha=0.9)
nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=7)
ax.set_title("TinyTransformerBlock — FX dataflow graph", fontsize=14, color="#1A1A1A")
ax.set_facecolor(CREAM)
ax.axis("off")

legend_handles = [Patch(facecolor=c, label=k) for k, c in color_for_op.items()]
ax.legend(handles=legend_handles, loc="lower left", frameon=False)
plt.tight_layout()
plt.show()

# TODO 2.2 — store the number of directed edges in `n_edges`
n_edges = None  # REPLACE  -- hint: G.number_of_edges()
print(f"Graph has {G.number_of_nodes()} nodes and {n_edges} edges.")


In [ ]:
# --- Checkpoint 2 -------------------------------------------------------
assert G.number_of_nodes() == sum(op_counts.values())
assert n_edges is not None and n_edges >= G.number_of_nodes() - 1
# Sanity: the output should have at least one incoming edge.
out_nodes = [n for n in gm.graph.nodes if n.op == "output"]
assert len(out_nodes) == 1
print(f"[OK] Graph visualized. {G.number_of_nodes()} nodes, {n_edges} edges.")


## Part 3 — Annotate every node with compute cost and memory footprint

A dataflow compiler needs two numbers for every node to make scheduling and tiling decisions:
- **FLOPs** — how much compute does this node do?
- **Bytes moved** — how much data must flow across the inputs and output?

Their ratio — *arithmetic intensity* (AI) — is the most important single number in performance analysis. `AI = FLOPs / bytes`. A node with AI > 10 is likely compute-bound; AI < 1 is memory-bound.

We'll do a simple first-order model:

| Op | FLOPs | Bytes (fp16) |
|---|---|---|
| `nn.Linear(d_in, d_out)` on `[B, S, d_in]` | `2 * B * S * d_in * d_out` | `2 * (B*S*d_in + d_in*d_out + B*S*d_out)` |
| `matmul([B,N,S,D] @ [B,N,D,S])` (Q·Kᵀ) | `2 * B * N * S * S * D` | `2 * (B*N*S*D + B*N*D*S + B*N*S*S)` |
| `matmul([B,N,S,S] @ [B,N,S,D])` (attn·V) | `2 * B * N * S * S * D` | `2 * (B*N*S*S + B*N*S*D + B*N*S*D)` |
| `LayerNorm`, `softmax`, `silu`, `mul`, `add` | ~0 FLOPs (relative), bytes = 2 × elementwise tensor size | (memory-bound) |

**Your task**: for each node in `gm.graph`, attach `node.meta["flops"]` and `node.meta["bytes"]` using the helper functions provided.


In [ ]:
# Helpers — these compute FLOPs and bytes given input/output shapes.
# You use them below; you don't need to modify them.

BYTES_PER_ELEM = 2  # fp16 assumption for the analysis

def linear_flops(in_features, out_features, batch_seq):
    # 2 * B * S * in_features * out_features (multiply + add per output elem)
    return 2 * batch_seq * in_features * out_features

def linear_bytes(in_features, out_features, batch_seq):
    # Input + weights + output, in fp16.
    elems = batch_seq * in_features + in_features * out_features + batch_seq * out_features
    return BYTES_PER_ELEM * elems

def matmul_flops(a_shape, b_shape):
    # a: (..., M, K), b: (..., K, N). FLOPs = 2 * M * N * K * batch_dims_product.
    bat = 1
    for d in a_shape[:-2]:
        bat *= d
    M, K = a_shape[-2], a_shape[-1]
    N    = b_shape[-1]
    return 2 * bat * M * N * K

def matmul_bytes(a_shape, b_shape, out_shape):
    def prod(s):
        p = 1
        for d in s: p *= d
        return p
    return BYTES_PER_ELEM * (prod(a_shape) + prod(b_shape) + prod(out_shape))

def elementwise_bytes(shape):
    # Read one tensor, write one tensor.
    p = 1
    for d in shape: p *= d
    return BYTES_PER_ELEM * 2 * p

# We run a concrete shape-propagation pass using torch.fx.passes.shape_prop.
from torch.fx.passes.shape_prop import ShapeProp
ShapeProp(gm).propagate(dummy)

# After ShapeProp, every node has node.meta["tensor_meta"] with .shape and .dtype.

# TODO 3.1 — walk every node. For each, compute FLOPs and bytes and attach
#             to node.meta["flops"] and node.meta["bytes"].
#
#             Follow these rules:
#               * placeholder / output / get_attr  -> flops=0, bytes=0
#               * call_module to nn.Linear         -> linear_flops / linear_bytes
#                   (use `gm.get_submodule(node.target)` to get the Linear;
#                    its in/out shape come from the input arg's shape + Linear.out_features)
#               * call_module to nn.LayerNorm      -> flops=0 (small), bytes = elementwise_bytes(output)
#               * call_function torch.matmul       -> matmul_flops / matmul_bytes
#                   (inspect args: both are FX nodes, with shapes in .meta["tensor_meta"].shape)
#               * everything else (softmax, silu, mul, add, transpose, reshape, truediv)
#                   -> flops=0, bytes = elementwise_bytes(output shape)
#
# The solution fills in roughly 25 lines. If yours is much longer, simplify.

for node in gm.graph.nodes:
    node.meta["flops"] = 0
    node.meta["bytes"] = 0
    # YOUR CODE HERE

# Print the annotated table
print(f"{'name':30s} {'op':15s} {'flops':>14s} {'bytes':>14s} {'AI':>8s}")
print("-" * 85)
for node in gm.graph.nodes:
    f = node.meta["flops"]
    b = node.meta["bytes"]
    ai = (f / b) if b > 0 else 0.0
    print(f"{node.name:30s} {node.op:15s} {f:14,d} {b:14,d} {ai:8.2f}")


In [ ]:
# --- Checkpoint 3 ------------------------------------------------------
# Every node should have flops and bytes keys set.
assert all("flops" in n.meta and "bytes" in n.meta for n in gm.graph.nodes)

# Total FLOPs should be dominated by the matmuls and Linears.
total_flops = sum(n.meta["flops"] for n in gm.graph.nodes)
total_bytes = sum(n.meta["bytes"] for n in gm.graph.nodes)

# Rough expected FLOP count for this block at HIDDEN=512 NUM_HEADS=8 FFN=2048 B=2 S=128:
#   4 QKVO Linears: 4 * 2 * 256 * 512 * 512               ~ 2.68e8
#   2 matmuls attn: 2 * 2 * 8 * 128 * 128 * 64            ~ 6.71e7
#   3 FFN Linears:  2 * (2 * 256 * 512 * 2048) * 2 + 1 up ~ 1.07e9
#   total around ~1.4e9
assert 5e8 < total_flops < 5e9, f"Total FLOPs look off: {total_flops:,}"
assert total_bytes > 0
print(f"[OK] Total FLOPs: {total_flops:,}  |  Total bytes: {total_bytes:,}")


## Part 4 — Classify each node

On a modern GPU (T4 / A100 / H100), the arithmetic-intensity "ridge" — the point where a workload stops being memory-bound and starts being compute-bound — is around **10 FLOPs/byte** for fp16 GEMM. Below that, peak compute is unreachable because the HBM bandwidth can't feed the cores.

We'll classify each node:
- **compute-bound**  if `AI >= 10`
- **memory-bound**   if `0 < AI < 10`
- **cheap**          if `bytes < 1e4` (tiny tensors — noise)

Later weeks will revisit these thresholds for Spyre (whose scratchpad bandwidth shifts the ridge) and TPU (whose VMEM + HBM hierarchy has *two* ridges).


In [ ]:
GPU_RIDGE_AI = 10.0
CHEAP_BYTES  = 1e4

# TODO 4.1 — classify every node. Attach node.meta["class"] in {"compute", "memory", "cheap"}.
# Then compute counts per class into `class_counts`.
class_counts = {"compute": 0, "memory": 0, "cheap": 0}
# YOUR CODE HERE

print("Node classification:")
for k, v in class_counts.items():
    print(f"  {k:10s}: {v}")


In [ ]:
# Re-draw the graph colored by classification (instead of op type)
class_color = {
    "compute": TERRACOTTA,    # heavy lifting
    "memory":  "#5C5C5C",     # memory-bound operators
    "cheap":   "#CDC5BA",     # low-cost
}
class_of = {n.name: n.meta.get("class", "cheap") for n in gm.graph.nodes}
colors2  = [class_color[class_of[name]] for name in G.nodes]

fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx_edges(G, pos, ax=ax, edge_color=SECONDARY, arrows=True, arrowsize=10, alpha=0.6)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors2, node_size=900, alpha=0.9)
nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=7)
ax.set_title("Classification: compute-bound (terracotta) vs memory-bound (gray) vs cheap",
             fontsize=14, color="#1A1A1A")
ax.set_facecolor(CREAM)
ax.axis("off")
legend_handles = [Patch(facecolor=c, label=k) for k, c in class_color.items()]
ax.legend(handles=legend_handles, loc="lower left", frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
# --- Checkpoint 4 ------------------------------------------------------
assert sum(class_counts.values()) == sum(op_counts.values()), \
    "Every node must be classified exactly once."
# Sanity: you should have at least a few compute-bound nodes (the Linears, matmuls).
assert class_counts["compute"] >= 3, f"Expected >= 3 compute-bound nodes; got {class_counts['compute']}"
# ... and some memory-bound / cheap ones (softmax, elementwise mul, add, etc).
assert class_counts["memory"] + class_counts["cheap"] >= 5
print(f"[OK] Classification complete: {class_counts}")


## Part 5 — Scheduling-bottleneck prediction (peer-reviewed)

Everything up to this point was measurement and bookkeeping. Now you reason.

A **dataflow accelerator** executes the graph you just built: each node fires when its inputs are ready. Three things tend to become bottlenecks on dataflow targets that *don't* hurt as much on a GPU:

1. **Shape transitions**: `transpose`, `reshape`, `permute` are free on a GPU (just a stride change, no data movement). On a dataflow target with fixed on-chip memory layouts, they can require *physical data reshuffling* — expensive.
2. **Softmax and other reductions along the wrong axis**: reductions across the SRAM-tiled axis stall the pipeline waiting for partial sums.
3. **Small ops between big ones**: GPUs amortize them via caches; dataflow targets may have to drain the pipeline.

---

### 5.1 — Which three nodes in YOUR graph would you predict are most likely to become scheduling bottlenecks on a generic dataflow target? Name them and justify.

Look at your classified graph. Pick three specific node names from *your* trace. For each, explain WHICH of the three categories above applies and WHY.

**YOUR ANALYSIS HERE**

---

### 5.2 — A dataflow compiler will likely FUSE many of these memory-bound nodes together. Identify one sequence in your graph where fusion looks obviously beneficial. What would a single fused node look like?

Hint: consider `ln2 -> ffn_gate -> silu -> mul -> ffn_up` or similar chains.

**YOUR ANALYSIS HERE**

---

### 5.3 — Predict what the Q·Kᵀ matmul's arithmetic intensity would look like at sequence length 4096 instead of 128. Would it shift from compute-bound to memory-bound, or the other way? Show the calculation.

This matters enormously for KV cache regimes. Week 4 will revisit it.

**YOUR ANALYSIS HERE**


## Part 6 — Update the Landscape Map

Week 2 adds a new row to the shared Accelerator Landscape Map: **"Graph representation complexity"** — how much work does each system's compiler do to get from the source PyTorch graph to something executable?

Fill in the row for PyTorch-on-GPU based on what you observed. We will fill in other accelerators in Weeks 3–4.


In [ ]:
# TODO 6.1 — add the Week-2 Landscape Map row.
landscape_row_w2 = {
    "accelerator":              None,  # e.g. f"GPU ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})"
    "graph_capture_mechanism":  None,  # e.g. "torch.fx symbolic_trace + TorchInductor dynamo (production)"
    "graph_size":               None,  # e.g. f"{G.number_of_nodes()} nodes / {n_edges} edges"
    "compile_time_work":        None,  # e.g. "operator fusion + Triton kernel codegen at first call"
    "scheduling_granularity":   None,  # e.g. "kernel-level (operators fused into CUDA kernels)"
    "compute_bound_nodes":      None,  # class_counts["compute"]
    "memory_bound_nodes":       None,  # class_counts["memory"]
}

for k, v in landscape_row_w2.items():
    print(f"  {k:28s}: {v}")


In [ ]:
# --- Checkpoint 6 ------------------------------------------------------
required = {"accelerator", "graph_capture_mechanism", "graph_size",
            "compile_time_work", "scheduling_granularity",
            "compute_bound_nodes", "memory_bound_nodes"}
assert set(landscape_row_w2.keys()) == required
assert all(v is not None for v in landscape_row_w2.values())
print("[OK] Week 2 Landscape Map row complete.")


In [ ]:
# --- Submission block -------------------------------------------------
import json as _json
submission = {
    "student_id":           "FILL-IN",
    "op_counts":            op_counts,
    "n_nodes":              G.number_of_nodes(),
    "n_edges":              n_edges,
    "total_flops":          sum(n.meta["flops"] for n in gm.graph.nodes),
    "total_bytes":          sum(n.meta["bytes"] for n in gm.graph.nodes),
    "class_counts":         class_counts,
    "landscape_row_w2":     landscape_row_w2,
}
with open("submission.json", "w") as _f:
    _json.dump(submission, _f, indent=2, default=str)
print("submission.json written.")


---

## Optional — Dataflow Deep Dive (Spyre hardware access required)

Open `../optional_spyre/lab2_spyre_extension.ipynb` to compile this same block with the Spyre compiler, dump its internal dataflow IR, and catalog what the compiler **changed** from your FX graph (fused ops, reordered nodes, inserted data-movement edges). You will compare its fusion decisions against your predictions in Part 5.2.

---

*Course: Dataflow Architectures for AI — Week 2 Lab, v1.0 · © 2026 Dr. Kaoutar El Maghraoui*
